# ACS ECG Detector — обучение на Google Colab (GPU T4)

Выполняет Этапы 4-5 (CNN + Multimodal) на бесплатном GPU T4.

## Подготовка (загрузить processed.zip в Google Drive перед запуском)
1. На локальном ПК запустить: `python scripts/pack_for_colab.py`
2. Загрузить `processed.zip` на Google Drive в папку `ecg-project/`
3. Убедиться, что репозиторий `acs-risk-ai` запушен на GitHub (или изменить команду git clone на свой URL)
4. Runtime -> Change runtime type -> T4 GPU

In [ ]:
# Ячейка 1: Подключить Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('OK Drive mounted')

In [ ]:
# Ячейка 2: Распаковать данные и конфиги
!cp "/content/drive/MyDrive/ecg-project/processed.zip" ./ || echo "processed.zip not found"
!unzip -o processed.zip
print('OK processed.zip extracted')

In [ ]:
# Ячейка 3: Клонировать репозиторий и установить зависимости
# Замените URL на свой GitHub-репозиторий
REPO_URL = "https://github.com/mlmamalyga83/acs-risk-ai.git"

import os
if not os.path.exists('acs-risk-ai'):
    !git clone {REPO_URL}

%cd acs-risk-ai
!pip install -r requirements.txt -q

# Скопировать данные и конфиги (из processed.zip, распакован в корне Colab)
!cp -r ../data/ ./ 2>/dev/null || echo "data already exists"
!cp -r ../config/ ./ 2>/dev/null || echo "config already exists"

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Ячейка 4: Проверить GPU и данные
import torch
assert torch.cuda.is_available(), "GPU не обнаружен! Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

import numpy as np
proc = 'data/processed'
print(f"Train files: {open(f'{proc}/X_train_manifest.txt').read().count(chr(10))} batches")
print(f"Val:   {np.load(f'{proc}/X_val.npy', mmap_mode='r').shape}")
print(f"Test:  {np.load(f'{proc}/X_test.npy', mmap_mode='r').shape}")
print(f"Train: {np.load(f'{proc}/y_train.npy').shape}, ACS={np.load(f'{proc}/y_train.npy').mean()*100:.1f}%")
print('OK all checks passed')

In [ ]:
# Ячейка 5: CNN обучение с Optuna (~6 часов на T4)
!python run.py --stage cnn --tune

In [ ]:
# Ячейка 6: Multimodal обучение + Ablation (~4 часа на T4)
!python run.py --stage multimodal --tune

In [ ]:
# Ячейка 7: Сохранить результаты на Google Drive
DRIVE_DIR = "/content/drive/MyDrive/ecg-project/"
!mkdir -p {DRIVE_DIR}/models/
!cp -r models/*.pt {DRIVE_DIR}/models/
!cp reports/model_comparison.csv {DRIVE_DIR}/ 2>/dev/null || echo "no comparison csv"
print('OK models saved to Google Drive')